# SGG Pipeline — Verification & Validation (V&V)

> **Enhanced notebook** — adds the missing V&V sections on top of the original accuracy + latency tests.

---

## What this notebook validates

| # | Stage | What it checks | Status |
|---|-------|----------------|--------|
| D | Smoke tests | Models load & forward-pass without error | original |
| A | CNN accuracy & latency | Top-1/3 accuracy, per-class P/R/F1, confusion matrix, ms/frame | original |
| B | SGG detection quality | Detection rate, avg objects/frame, avg confidence, nodes & edges | original |
| C | Full pipeline latency | CNN + YOLO + graph build ms/video | original |
| **E** | **SGG graph structural V&V** | **Triplet completeness, relation-type coverage, node-label validity, graph connectivity, duplicate-edge & self-loop checks** | NEW |
| **F** | **SGG semantic V&V** | **Hazard-class recall, vocabulary compliance, confidence-threshold sensitivity, false-positive audit** | NEW |
| **G** | **Regression / unit tests (unittest)** | **14 deterministic unit tests covering every helper function** | NEW |
| **H** | **Matplotlib V&V dashboard** | **Confusion matrix heat-map, P/R/F1 bars, latency CDF, detections/video, node/edge scatter, conf histogram** | NEW |
| **J** | **Final pass/fail verdict** | **Structured JSON + human-readable PASS/WARN/FAIL per criterion** | NEW |

In [ ]:
!pip install ultralytics matplotlib seaborn --quiet

In [ ]:
# ================================================================
#  CELL 1 - CONFIGURATION  (edit paths here)
# ================================================================
import torch

ENV_MODEL_PATH    = "/content/drive/MyDrive/Other/places365_environment_model_new.pth"
CLASSES           = ["classroom", "home", "office"]
TEST_DIR          = "/content/drive/MyDrive/Colab Notebooks/Data"
SINGLE_VIDEO      = None
SINGLE_LABEL      = None
YOLO_WEIGHTS      = "yolov8s-world.pt"
NO_SGG            = False
REPORT_PATH       = "validation_report.json"
NUM_SAMPLE_FRAMES = 5

DEFAULT_SGG_VOCAB = [
    "person", "gun", "knife",
    "fire", "flames", "smoke", "fireplace", "burning",
    "person falling", "chair", "desk", "laptop", "projector",
]

# V&V pass/fail thresholds
THRESH_CNN_ACCURACY     = 0.70
THRESH_DETECTION_RATE   = 0.50
THRESH_PIPELINE_MS      = 2000
THRESH_TRIPLET_COMPLETE = 0.80
THRESH_HAZARD_RECALL    = 0.60
YOLO_CONF_THRESHOLD     = 0.25

print("Configuration set.")
print(f"  Model  : {ENV_MODEL_PATH}")
print(f"  Classes: {CLASSES}")
print(f"  Device : {'GPU' if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# ================================================================
#  CELL 2 - IMPORTS
# ================================================================
import os, sys, cv2, time, json, warnings, unittest, io
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
from collections import defaultdict
from typing import List, Dict, Tuple, Optional

warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
from torchvision import transforms
import torchvision.models as models
from PIL import Image

try:
    from ultralytics import YOLOWorld
    YOLO_AVAILABLE = True
    print("ultralytics / YOLO-World available")
except ImportError:
    YOLO_AVAILABLE = False
    print("ultralytics not installed - SGG tests will be skipped.")

print("All imports successful.")

In [ ]:
# ================================================================
#  CELL 3 - COLOUR HELPERS
# ================================================================
class C:
    HEADER = "\033[95m"; BLUE  = "\033[94m"; CYAN  = "\033[96m"
    GREEN  = "\033[92m"; YELLOW= "\033[93m"; RED   = "\033[91m"
    BOLD   = "\033[1m";  END   = "\033[0m"

def banner(title):
    line = "=" * 60
    print(f"\n{C.BOLD}{C.BLUE}{line}{C.END}")
    print(f"{C.BOLD}{C.BLUE}  {title}{C.END}")
    print(f"{C.BOLD}{C.BLUE}{line}{C.END}")

def ok(msg):   print(f"  {C.GREEN}PASS  {msg}{C.END}")
def warn(msg): print(f"  {C.YELLOW}WARN  {msg}{C.END}")
def fail(msg): print(f"  {C.RED}FAIL  {msg}{C.END}")
def info(msg): print(f"  {C.CYAN}INFO  {msg}{C.END}")

print("Helper functions defined.")

In [ ]:
# ================================================================
#  CELL 4 - MODEL LOADERS
# ================================================================
def load_env_model(model_path: str, num_classes: int = 3) -> tuple:
    """Load the fine-tuned ResNet18 environment classifier."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model  = models.resnet18(num_classes=365)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device).eval()
    return model, device

def load_sgg_model(weights: str = YOLO_WEIGHTS, vocab: list = None):
    """Load YOLO-World and configure vocabulary."""
    if not YOLO_AVAILABLE:
        return None
    model = YOLOWorld(weights)
    if vocab:
        model.set_classes(vocab)
    return model

PREPROCESS = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

print("Model loader functions defined.")

In [ ]:
# ================================================================
#  CELL 5 - UTILITY FUNCTIONS
# ================================================================
def calculate_iou(boxA, boxB):
    """Intersection over Union for two bounding boxes [x1,y1,x2,y2]."""
    xA = max(boxA[0], boxB[0]); yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2]); yB = min(boxA[3], boxB[3])
    interArea = max(0, xB - xA) * max(0, yB - yA)
    areaA = (boxA[2]-boxA[0]) * (boxA[3]-boxA[1])
    areaB = (boxB[2]-boxB[0]) * (boxB[3]-boxB[1])
    return interArea / float(areaA + areaB - interArea + 1e-5)

def sample_frames(video_path: str, n: int = NUM_SAMPLE_FRAMES):
    """Return list of (frame_index, bgr_frame) for n evenly-spaced frames."""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Cannot open video: {video_path}")
    total   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps_val = cap.get(cv2.CAP_PROP_FPS) or 30.0
    indices = set(np.linspace(0, total - 1, n, dtype=int).tolist())
    frames  = []
    for i in range(total):
        ret, frame = cap.read()
        if not ret: break
        if i in indices:
            frames.append((i, frame))
    cap.release()
    return frames, fps_val, total

class LatencyTimer:
    """Accumulates timing measurements and computes statistics."""
    def __init__(self, name: str):
        self.name  = name
        self.times = []

    def measure(self, fn, *args, **kwargs):
        t0     = time.perf_counter()
        result = fn(*args, **kwargs)
        self.times.append((time.perf_counter() - t0) * 1000)
        return result

    def stats(self) -> dict:
        if not self.times:
            return {"n": 0, "mean_ms": 0, "min_ms": 0, "max_ms": 0, "p95_ms": 0}
        arr = np.array(self.times)
        return {
            "n":       len(arr),
            "mean_ms": round(float(np.mean(arr)),            2),
            "min_ms":  round(float(np.min(arr)),             2),
            "max_ms":  round(float(np.max(arr)),             2),
            "p95_ms":  round(float(np.percentile(arr, 95)),  2),
        }

    def report(self):
        s = self.stats()
        info(f"{self.name:30s}  mean={s['mean_ms']:7.1f}ms  min={s['min_ms']:7.1f}ms  max={s['max_ms']:7.1f}ms  p95={s['p95_ms']:7.1f}ms  (n={s['n']})")

print("Utility functions defined.")

In [ ]:
# ================================================================
#  CELL 6 - STAGE A: CNN ENVIRONMENT CLASSIFIER
# ================================================================
def validate_env_model(env_model, device, test_videos: list, classes: list) -> dict:
    banner("STAGE A -- Environment CNN Accuracy & Latency")
    if not test_videos:
        warn("No labelled test videos provided. Skipping accuracy test.")
        return {}

    timer = LatencyTimer("CNN inference / frame")
    y_true, y_pred, y_conf = [], [], []
    y_top3_correct = []   # NEW: track top-3
    video_results  = []

    for video_path, true_label in test_videos:
        if true_label not in classes:
            warn(f"Label '{true_label}' not in class list -- skipping {video_path}")
            continue
        true_idx = classes.index(true_label)
        try:
            frames, _, _ = sample_frames(video_path)
        except IOError as e:
            fail(str(e)); continue

        accumulated = torch.zeros(len(classes)).to(device)

        for _, bgr in frames:
            rgb    = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            pil    = Image.fromarray(rgb)
            tensor = PREPROCESS(pil).unsqueeze(0).to(device)
            def _infer():
                with torch.no_grad():
                    out   = env_model(tensor)
                    probs = torch.nn.functional.softmax(out[0], dim=0)
                return probs
            probs = timer.measure(_infer)
            accumulated += probs

        avg_probs      = accumulated / len(frames)
        conf, pred_idx = torch.max(avg_probs, 0)
        top3_indices   = torch.topk(avg_probs, min(3, len(classes))).indices.tolist()

        y_true.append(true_idx)
        y_pred.append(pred_idx.item())
        y_conf.append(conf.item())
        y_top3_correct.append(true_idx in top3_indices)

        correct = (pred_idx.item() == true_idx)
        sc      = C.GREEN if correct else C.RED
        sym     = "PASS" if correct else "FAIL"
        print(f"  {sc}[{sym}]  {Path(video_path).name:35s}  true={true_label:12s}  pred={classes[pred_idx.item()]:12s}  conf={conf.item():.3f}{C.END}")

        video_results.append({
            "video":      video_path,
            "true_label": true_label,
            "pred_label": classes[pred_idx.item()],
            "confidence": round(conf.item(), 4),
            "correct":    correct,
        })

    n_total   = len(y_true)
    n_correct = sum(1 for t, p in zip(y_true, y_pred) if t == p)
    accuracy  = n_correct / n_total if n_total else 0.0
    top3_acc  = sum(y_top3_correct) / n_total if n_total else 0.0

    banner("ENV CNN -- Results Summary")
    ok(f"Top-1 Accuracy : {n_correct}/{n_total} = {accuracy*100:.1f}%")
    ok(f"Top-3 Accuracy : {sum(y_top3_correct)}/{n_total} = {top3_acc*100:.1f}%")
    timer.report()

    per_class = {}
    for ci, cls in enumerate(classes):
        tp = sum(1 for t, p in zip(y_true, y_pred) if t == ci and p == ci)
        fp = sum(1 for t, p in zip(y_true, y_pred) if t != ci and p == ci)
        fn = sum(1 for t, p in zip(y_true, y_pred) if t == ci and p != ci)
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall    = tp / (tp + fn) if (tp + fn) else 0.0
        f1        = (2 * precision * recall / (precision + recall) if (precision + recall) else 0.0)
        per_class[cls] = {
            "precision": round(precision, 4),
            "recall":    round(recall,    4),
            "f1":        round(f1,        4),
            "support":   sum(1 for t in y_true if t == ci),
        }
        print(f"  {cls:12s}  P={precision:.3f}  R={recall:.3f}  F1={f1:.3f}  (n={per_class[cls]['support']})")

    matrix = np.zeros((len(classes), len(classes)), dtype=int)
    for t, p in zip(y_true, y_pred):
        matrix[t][p] += 1

    print("\n  Confusion Matrix  (rows=true, cols=pred)")
    header = "           " + "  ".join(f"{c[:8]:>8}" for c in classes)
    print(f"  {header}")
    for i, row in enumerate(matrix):
        row_str = "  ".join(f"{v:>8}" for v in row)
        print(f"  {classes[i][:10]:>10s}  {row_str}")

    return {
        "accuracy":         round(accuracy, 4),
        "top3_accuracy":    round(top3_acc, 4),
        "n_videos":         n_total,
        "n_correct":        n_correct,
        "per_class":        per_class,
        "confusion_matrix": matrix.tolist(),
        "y_true":           y_true,
        "y_pred":           y_pred,
        "latency":          timer.stats(),
        "latency_times":    timer.times,
        "video_results":    video_results,
    }

print("Stage A function defined.")

In [ ]:
# ================================================================
#  CELL 7 - STAGE B: SGG YOLO-WORLD DETECTION QUALITY
# ================================================================
def validate_sgg(sgg_model, test_videos: list) -> dict:
    banner("STAGE B -- SGG YOLO-World Detection Quality & Latency")
    if not YOLO_AVAILABLE or sgg_model is None:
        warn("YOLO-World not available -- skipping SGG validation."); return {}
    if not test_videos:
        warn("No test videos -- skipping SGG validation.");          return {}

    timer_yolo = LatencyTimer("YOLO-World inference / frame")
    timer_sgg  = LatencyTimer("SGG graph build / frame")

    total_frames_checked   = 0
    frames_with_detections = 0
    all_obj_counts         = []
    all_confidences        = []
    all_node_counts        = []
    all_edge_counts        = []
    video_sgg_results      = []

    for video_path, _ in test_videos:
        try:
            frames, _, _ = sample_frames(video_path)
        except IOError as e:
            fail(str(e)); continue

        video_nodes = video_edges = video_detections = 0

        for _, bgr in frames:
            total_frames_checked += 1
            results = timer_yolo.measure(lambda f=bgr: sgg_model.predict(f, verbose=False))
            n_det   = len(results[0].boxes) if results else 0
            all_obj_counts.append(n_det)

            if n_det > 0:
                frames_with_detections += 1
                confs = results[0].boxes.conf.cpu().numpy()
                all_confidences.extend(confs.tolist())
                video_detections += n_det

            def _build_graph(res=results):
                nodes, edges = [], []
                if not res or len(res[0].boxes) == 0:
                    return nodes, edges
                boxes_arr = res[0].boxes.xyxy.cpu().numpy()
                cls_arr   = res[0].boxes.cls.cpu().numpy()
                conf_arr  = res[0].boxes.conf.cpu().numpy()
                names     = res[0].names
                for i, cid in enumerate(cls_arr):
                    if conf_arr[i] >= YOLO_CONF_THRESHOLD:
                        label = names[int(cid)].replace("_"," ").title()
                        nodes.append({"id": f"n{i}", "label": label, "conf": round(float(conf_arr[i]), 3)})
                ids = list(range(len(nodes)))
                for i in ids:
                    for j in ids:
                        if i >= j: continue
                        iou = calculate_iou(boxes_arr[i], boxes_arr[j])
                        if   iou > 0.05: edges.append({"src": f"n{i}", "tgt": f"n{j}", "rel": "interacting_with"})
                        elif iou > 0.0:  edges.append({"src": f"n{i}", "tgt": f"n{j}", "rel": "adjacent_to"})
                return nodes, edges

            nodes, edges = timer_sgg.measure(_build_graph)
            all_node_counts.append(len(nodes))
            all_edge_counts.append(len(edges))
            video_nodes += len(nodes)
            video_edges += len(edges)

        info(f"{Path(video_path).name:35s}  detections={video_detections:3d}  nodes={video_nodes:3d}  edges={video_edges:3d}")
        video_sgg_results.append({
            "video":       video_path,
            "detections":  video_detections,
            "total_nodes": video_nodes,
            "total_edges": video_edges,
        })

    detection_rate = frames_with_detections / total_frames_checked if total_frames_checked else 0
    avg_objs  = float(np.mean(all_obj_counts))  if all_obj_counts  else 0
    avg_conf  = float(np.mean(all_confidences)) if all_confidences else 0
    avg_nodes = float(np.mean(all_node_counts)) if all_node_counts else 0
    avg_edges = float(np.mean(all_edge_counts)) if all_edge_counts else 0

    banner("SGG -- Results Summary")
    ok(f"Detection rate   : {frames_with_detections}/{total_frames_checked} = {detection_rate*100:.1f}%")
    ok(f"Avg objects/frame: {avg_objs:.2f}")
    ok(f"Avg confidence   : {avg_conf:.3f}")
    ok(f"Avg nodes/frame  : {avg_nodes:.2f}")
    ok(f"Avg edges/frame  : {avg_edges:.2f}")
    timer_yolo.report(); timer_sgg.report()

    return {
        "detection_rate":        round(detection_rate, 4),
        "frames_checked":        total_frames_checked,
        "avg_objects_per_frame": round(avg_objs,  4),
        "avg_confidence":        round(avg_conf,  4),
        "avg_nodes_per_frame":   round(avg_nodes, 4),
        "avg_edges_per_frame":   round(avg_edges, 4),
        "yolo_latency":          timer_yolo.stats(),
        "yolo_latency_times":    timer_yolo.times,
        "sgg_latency":           timer_sgg.stats(),
        "video_results":         video_sgg_results,
        "all_node_counts":       all_node_counts,
        "all_edge_counts":       all_edge_counts,
    }

print("Stage B function defined.")

In [ ]:
# ================================================================
#  CELL 8 - STAGE C: FULL PIPELINE LATENCY
# ================================================================
def validate_pipeline_latency(env_model, device, sgg_model, test_videos: list, classes: list) -> dict:
    banner("STAGE C -- Full Pipeline End-to-End Latency")
    timer_full = LatencyTimer("Full pipeline / video")
    results    = []

    for video_path, true_label in test_videos:
        def _run_pipeline(vp=video_path):
            frames, _, _ = sample_frames(vp)
            accumulated  = torch.zeros(len(classes)).to(device)
            max_det = -1
            for _, bgr in frames:
                if sgg_model:
                    res = sgg_model.predict(bgr, verbose=False)
                    n   = len(res[0].boxes) if res else 0
                    if n > max_det: max_det = n
                rgb    = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
                tensor = PREPROCESS(Image.fromarray(rgb)).unsqueeze(0).to(device)
                with torch.no_grad():
                    out   = env_model(tensor)
                    probs = torch.nn.functional.softmax(out[0], dim=0)
                    accumulated += probs
            avg  = accumulated / len(frames)
            conf, idx = torch.max(avg, 0)
            return classes[idx.item()], conf.item()

        predicted, confidence = timer_full.measure(_run_pipeline)
        correct = (predicted == true_label)
        last_t  = timer_full.times[-1]
        sc      = C.GREEN if correct else C.RED
        sym     = "PASS" if correct else "FAIL"
        print(f"  {sc}[{sym}]  {Path(video_path).name:35s}  pred={predicted:12s}  conf={confidence:.3f}  time={last_t:.1f}ms{C.END}")
        results.append({
            "video":      video_path,  "true_label": true_label,
            "pred_label": predicted,   "confidence": round(confidence, 4),
            "correct":    correct,     "latency_ms": round(last_t, 2),
        })

    banner("Pipeline Latency Summary")
    timer_full.report()
    stats = timer_full.stats()
    if   stats["mean_ms"] < 500:  ok(f"Pipeline FAST   -- avg {stats['mean_ms']:.0f}ms/video")
    elif stats["mean_ms"] < 2000: warn(f"Pipeline MODERATE -- avg {stats['mean_ms']:.0f}ms/video")
    else:                         warn(f"Pipeline SLOW   -- avg {stats['mean_ms']:.0f}ms/video")

    return {"latency": stats, "latency_times": timer_full.times, "video_results": results}

print("Stage C function defined.")

In [ ]:
# ================================================================
#  CELL 9 - STAGE D: SMOKE TESTS
# ================================================================
def run_smoke_tests(env_model, device, sgg_model, classes: list) -> dict:
    banner("STAGE D -- Smoke Tests (model sanity checks)")
    passed = failed = 0

    for test_name, test_fn in [
        ("CNN forward pass shape", lambda: (
            lambda out: (ok(f"CNN forward pass -- output shape {tuple(out.shape)}"), True)[1]
        )(env_model(torch.randn(1, 3, 224, 224).to(device)))),
    ]:
        pass  # placeholder replaced by explicit tests below

    # T1 - CNN forward pass shape
    try:
        dummy = torch.randn(1, 3, 224, 224).to(device)
        with torch.no_grad(): out = env_model(dummy)
        assert out.shape == (1, len(classes))
        ok(f"CNN forward pass -- output shape {tuple(out.shape)}"); passed += 1
    except Exception as e:
        fail(f"CNN forward pass failed: {e}"); failed += 1

    # T2 - CNN softmax sums to 1
    try:
        dummy = torch.randn(1, 3, 224, 224).to(device)
        with torch.no_grad():
            out   = env_model(dummy)
            probs = torch.softmax(out[0], dim=0)
        assert abs(probs.sum().item() - 1.0) < 1e-4
        ok(f"CNN softmax sanity -- probs sum to {probs.sum().item():.6f}"); passed += 1
    except Exception as e:
        fail(f"CNN probability check failed: {e}"); failed += 1

    # T3 - CNN latency warm-up
    try:
        dummy = torch.randn(1, 3, 224, 224).to(device)
        t0 = time.perf_counter()
        for _ in range(10):
            with torch.no_grad(): env_model(dummy)
        avg_ms = (time.perf_counter() - t0) / 10 * 1000
        ok(f"CNN single-frame latency -- {avg_ms:.1f}ms avg over 10 runs"); passed += 1
    except Exception as e:
        fail(f"CNN latency test failed: {e}"); failed += 1

    if sgg_model:
        # T4 - YOLO blank frame
        try:
            blank = np.zeros((480, 640, 3), dtype=np.uint8)
            t0    = time.perf_counter()
            res   = sgg_model.predict(blank, verbose=False)
            ms    = (time.perf_counter() - t0) * 1000
            n_det = len(res[0].boxes) if res else 0
            ok(f"YOLO blank frame -- {n_det} detections in {ms:.1f}ms"); passed += 1
        except Exception as e:
            fail(f"YOLO blank frame failed: {e}"); failed += 1

        # T5 - YOLO noise frame
        try:
            noise = np.random.randint(0, 255, (480, 640, 3), dtype=np.uint8)
            res   = sgg_model.predict(noise, verbose=False)
            ok(f"YOLO noise frame -- {len(res[0].boxes)} detections"); passed += 1
        except Exception as e:
            fail(f"YOLO noise frame failed: {e}"); failed += 1
    else:
        warn("YOLO smoke tests skipped (model not loaded)")

    # T6 - IOU edge cases
    try:
        iou_overlap = calculate_iou([0,0,10,10], [5,5,15,15])
        iou_none    = calculate_iou([0,0,5,5],   [10,10,20,20])
        assert 0 < iou_overlap < 1
        assert iou_none == 0.0
        ok(f"IOU helper -- partial={iou_overlap:.3f}, none={iou_none:.3f}"); passed += 1
    except Exception as e:
        fail(f"IOU test failed: {e}"); failed += 1

    banner(f"Smoke Tests -- {passed} passed, {failed} failed")
    if failed == 0: ok("All smoke tests passed -- pipeline is ready for integration")
    else:           warn(f"{failed} test(s) failed -- review before integration")

    return {"passed": passed, "failed": failed}

print("Stage D function defined.")

In [ ]:
# ================================================================
#  CELL 10 - NEW: STAGE E -- SGG GRAPH STRUCTURAL V&V
# ================================================================
"""
Validates that scene graphs meet structural requirements:
  E1  Triplet completeness  -- every edge src and tgt exist as nodes
  E2  Node-label validity   -- every node label is in DEFAULT_SGG_VOCAB
  E3  Duplicate-edge check  -- no (src, tgt, rel) triple appears twice in one frame
  E4  Self-loop check       -- no edge where src == tgt
  E5  Relation coverage     -- at least 2 distinct relation types found
  E6  Graph connectivity    -- ratio of frames that have at least one edge
"""

def validate_sgg_structure(sgg_model, test_videos: list, vocab: list = None) -> dict:
    banner("STAGE E -- SGG Graph Structural V&V")

    if not YOLO_AVAILABLE or sgg_model is None:
        warn("YOLO-World not available -- skipping structural V&V.")
        return {}
    if not test_videos:
        warn("No test videos -- skipping structural V&V.")
        return {}

    allowed_labels = {v.replace("_"," ").title() for v in (vocab or DEFAULT_SGG_VOCAB)}

    total_frames         = 0
    total_triplets       = 0
    complete_triplets    = 0
    invalid_label_count  = 0
    duplicate_edge_count = 0
    self_loop_count      = 0
    connected_frames     = 0
    all_relations        = set()
    per_video_issues     = []

    for video_path, _ in test_videos:
        try:
            frames, _, _ = sample_frames(video_path)
        except IOError as e:
            fail(str(e)); continue

        video_issues = []

        for fi, (_, bgr) in enumerate(frames):
            total_frames += 1
            results = sgg_model.predict(bgr, verbose=False)
            if not results or len(results[0].boxes) == 0:
                continue

            boxes_arr = results[0].boxes.xyxy.cpu().numpy()
            cls_arr   = results[0].boxes.cls.cpu().numpy()
            conf_arr  = results[0].boxes.conf.cpu().numpy()
            names     = results[0].names

            nodes = {}
            for i, cid in enumerate(cls_arr):
                if conf_arr[i] >= YOLO_CONF_THRESHOLD:
                    label = names[int(cid)].replace("_"," ").title()
                    nodes[f"n{i}"] = label
                    if label not in allowed_labels:
                        invalid_label_count += 1
                        video_issues.append(f"frame {fi}: unknown label '{label}'")

            edges      = []
            seen_edges = set()
            ids        = list(range(len(nodes)))
            for i in ids:
                for j in ids:
                    if i >= j: continue
                    src = f"n{i}"; tgt = f"n{j}"
                    iou = calculate_iou(boxes_arr[i], boxes_arr[j])
                    rel = None
                    if   iou > 0.05: rel = "interacting_with"
                    elif iou > 0.0:  rel = "adjacent_to"
                    if rel is None: continue
                    edges.append({"src": src, "tgt": tgt, "rel": rel})
                    all_relations.add(rel)

            for edge in edges:
                total_triplets += 1
                # E1 completeness
                if edge["src"] in nodes and edge["tgt"] in nodes:
                    complete_triplets += 1
                else:
                    video_issues.append(f"frame {fi}: dangling edge {edge}")
                # E3 duplicate
                key = (edge["src"], edge["tgt"], edge["rel"])
                if key in seen_edges:
                    duplicate_edge_count += 1
                    video_issues.append(f"frame {fi}: duplicate edge {key}")
                seen_edges.add(key)
                # E4 self-loop
                if edge["src"] == edge["tgt"]:
                    self_loop_count += 1
                    video_issues.append(f"frame {fi}: self-loop on {edge['src']}")

            if len(edges) > 0:
                connected_frames += 1

        per_video_issues.append({"video": video_path, "issues": video_issues[:10]})

    triplet_completeness = complete_triplets / total_triplets if total_triplets else 1.0
    connectivity_ratio   = connected_frames  / total_frames   if total_frames   else 0.0
    relation_types       = list(all_relations)

    banner("Stage E -- Structural V&V Results")
    col = C.GREEN if triplet_completeness >= THRESH_TRIPLET_COMPLETE else C.RED
    print(f"  {col}E1 Triplet completeness : {complete_triplets}/{total_triplets} = {triplet_completeness*100:.1f}%{C.END}")
    col = C.GREEN if invalid_label_count == 0 else C.YELLOW
    print(f"  {col}E2 Invalid node labels  : {invalid_label_count}{C.END}")
    col = C.GREEN if duplicate_edge_count == 0 else C.YELLOW
    print(f"  {col}E3 Duplicate edges      : {duplicate_edge_count}{C.END}")
    col = C.GREEN if self_loop_count == 0 else C.RED
    print(f"  {col}E4 Self-loop edges      : {self_loop_count}{C.END}")
    col = C.GREEN if len(relation_types) >= 2 else C.YELLOW
    print(f"  {col}E5 Relation types found : {relation_types}{C.END}")
    col = C.GREEN if connectivity_ratio >= 0.5 else C.YELLOW
    print(f"  {col}E6 Graph connectivity   : {connected_frames}/{total_frames} = {connectivity_ratio*100:.1f}%{C.END}")

    return {
        "triplet_completeness":  round(triplet_completeness, 4),
        "total_triplets":        total_triplets,
        "complete_triplets":     complete_triplets,
        "invalid_label_count":   invalid_label_count,
        "duplicate_edge_count":  duplicate_edge_count,
        "self_loop_count":       self_loop_count,
        "relation_types":        relation_types,
        "connectivity_ratio":    round(connectivity_ratio, 4),
        "per_video_issues":      per_video_issues,
    }

print("Stage E function defined.")

In [ ]:
# ================================================================
#  CELL 11 - NEW: STAGE F -- SGG SEMANTIC V&V
# ================================================================
"""
  F1  Hazard-class detection  -- gun/fire/knife/smoke found in relevant scenes
  F2  Vocabulary compliance   -- all labels are in the declared vocab
  F3  Confidence distribution -- mean, median, min, max of all detection confs
  F4  False-positive audit    -- fraction of detections below threshold
  F5  Threshold sensitivity   -- how detection rate shifts with conf cutoff
"""

HAZARD_CLASSES = {"Gun", "Knife", "Fire", "Flames", "Smoke", "Burning", "Person Falling"}

def validate_sgg_semantics(sgg_model, test_videos: list, vocab: list = None) -> dict:
    banner("STAGE F -- SGG Semantic V&V")

    if not YOLO_AVAILABLE or sgg_model is None:
        warn("YOLO-World not available -- skipping semantic V&V.")
        return {}
    if not test_videos:
        warn("No test videos -- skipping semantic V&V.")
        return {}

    allowed_labels     = {v.replace("_"," ").title() for v in (vocab or DEFAULT_SGG_VOCAB)}
    all_conf_scores    = []
    all_labels         = []
    low_conf_count     = 0
    total_det          = 0
    hazard_found       = set()
    raw_frame_confs    = []

    for video_path, _ in test_videos:
        try:
            frames, _, _ = sample_frames(video_path)
        except IOError as e:
            fail(str(e)); continue

        for _, bgr in frames:
            results = sgg_model.predict(bgr, verbose=False)
            if not results or len(results[0].boxes) == 0:
                raw_frame_confs.append(np.array([]))
                continue
            conf_arr = results[0].boxes.conf.cpu().numpy()
            cls_arr  = results[0].boxes.cls.cpu().numpy()
            names    = results[0].names
            raw_frame_confs.append(conf_arr)

            for i, cid in enumerate(cls_arr):
                label  = names[int(cid)].replace("_"," ").title()
                c_val  = float(conf_arr[i])
                total_det      += 1
                all_conf_scores.append(c_val)
                all_labels.append(label)
                if c_val < YOLO_CONF_THRESHOLD:
                    low_conf_count += 1
                if label in HAZARD_CLASSES:
                    hazard_found.add(label)

    detected_set     = set(all_labels)
    out_of_vocab     = detected_set - allowed_labels
    vocab_compliance = 1.0 - (len(out_of_vocab) / len(detected_set) if detected_set else 0.0)
    fp_rate          = low_conf_count / total_det if total_det else 0.0

    # F5 threshold sensitivity sweep
    thresholds      = np.arange(0.05, 0.95, 0.05)
    thresh_det_rate = []
    for th in thresholds:
        frames_det = sum(1 for fc in raw_frame_confs if np.any(fc >= th))
        thresh_det_rate.append(frames_det / len(raw_frame_confs) if raw_frame_confs else 0.0)

    banner("Stage F -- Semantic V&V Results")
    info(f"F1 Hazard classes detected : {hazard_found or 'none'}")
    col = C.GREEN if vocab_compliance >= 0.95 else C.YELLOW
    print(f"  {col}F2 Vocab compliance : {vocab_compliance*100:.1f}%{C.END}")
    if out_of_vocab: warn(f"   Out-of-vocab labels: {out_of_vocab}")
    if all_conf_scores:
        info(f"F3 Confidence dist : mean={np.mean(all_conf_scores):.3f}  "
             f"median={np.median(all_conf_scores):.3f}  "
             f"min={np.min(all_conf_scores):.3f}  max={np.max(all_conf_scores):.3f}")
    col = C.GREEN if fp_rate < 0.20 else C.YELLOW
    print(f"  {col}F4 Low-conf FP rate : {low_conf_count}/{total_det} = {fp_rate*100:.1f}%{C.END}")
    info(f"F5 Threshold sensitivity logged across {len(thresholds)} thresholds")

    return {
        "hazard_classes_detected": list(hazard_found),
        "vocab_compliance":        round(vocab_compliance, 4),
        "out_of_vocab_labels":     list(out_of_vocab),
        "conf_mean":               round(float(np.mean(all_conf_scores)),   4) if all_conf_scores else None,
        "conf_median":             round(float(np.median(all_conf_scores)), 4) if all_conf_scores else None,
        "low_conf_fp_rate":        round(fp_rate, 4),
        "threshold_sensitivity": {
            "thresholds":      [round(t, 2) for t in thresholds.tolist()],
            "detection_rates": [round(r, 4) for r in thresh_det_rate],
        },
        "all_conf_scores": all_conf_scores,
    }

print("Stage F function defined.")

In [ ]:
# ================================================================
#  CELL 12 - NEW: STAGE G -- UNITTEST REGRESSION SUITE
#  14 deterministic unit tests on synthetic data (no models needed)
# ================================================================

class TestIoU(unittest.TestCase):
    """Tests for calculate_iou helper."""

    def test_full_overlap(self):
        """Identical boxes must return IOU ~= 1.0"""
        self.assertAlmostEqual(calculate_iou([0,0,10,10], [0,0,10,10]), 1.0, places=2)

    def test_no_overlap(self):
        """Non-overlapping boxes must return 0.0"""
        self.assertEqual(calculate_iou([0,0,5,5], [10,10,20,20]), 0.0)

    def test_partial_overlap_between_zero_and_one(self):
        """Partial overlap must be strictly between 0 and 1"""
        iou = calculate_iou([0,0,10,10], [5,5,15,15])
        self.assertGreater(iou, 0)
        self.assertLess(iou, 1)

    def test_symmetry(self):
        """IOU(A,B) must equal IOU(B,A)"""
        a = calculate_iou([0,0,10,10], [5,5,15,15])
        b = calculate_iou([5,5,15,15], [0,0,10,10])
        self.assertAlmostEqual(a, b, places=5)

    def test_zero_area_no_crash(self):
        """Zero-area box must not crash and return >= 0"""
        iou = calculate_iou([5,5,5,5], [0,0,10,10])
        self.assertGreaterEqual(iou, 0.0)


class TestLatencyTimer(unittest.TestCase):
    """Tests for LatencyTimer statistics."""

    def setUp(self):
        self.timer = LatencyTimer("test")
        for v in [10.0, 20.0, 30.0, 40.0, 50.0]:
            self.timer.times.append(v)

    def test_mean_correct(self):
        self.assertAlmostEqual(self.timer.stats()["mean_ms"], 30.0, places=1)

    def test_min_correct(self):
        self.assertAlmostEqual(self.timer.stats()["min_ms"], 10.0, places=1)

    def test_max_correct(self):
        self.assertAlmostEqual(self.timer.stats()["max_ms"], 50.0, places=1)

    def test_empty_timer_returns_zeros(self):
        t = LatencyTimer("empty")
        s = t.stats()
        self.assertEqual(s["n"],       0)
        self.assertEqual(s["mean_ms"], 0)

    def test_measure_returns_fn_result(self):
        t      = LatencyTimer("fn")
        result = t.measure(lambda: 42)
        self.assertEqual(result, 42)
        self.assertEqual(len(t.times), 1)


class TestGraphStructure(unittest.TestCase):
    """Tests for SGG graph structural properties."""

    def _triplet_completeness(self, nodes, edges):
        node_ids = {n["id"] for n in nodes}
        complete = sum(1 for e in edges
                       if e["src"] in node_ids and e["tgt"] in node_ids)
        return complete, len(edges)

    def test_all_triplets_complete(self):
        nodes = [{"id": "n0"}, {"id": "n1"}]
        edges = [{"src": "n0", "tgt": "n1", "rel": "adjacent_to"}]
        c, t  = self._triplet_completeness(nodes, edges)
        self.assertEqual(c, t)

    def test_dangling_edge_detected(self):
        nodes = [{"id": "n0"}]
        edges = [{"src": "n0", "tgt": "n99", "rel": "adjacent_to"}]
        c, t  = self._triplet_completeness(nodes, edges)
        self.assertLess(c, t)

    def test_no_self_loops_in_valid_graph(self):
        edges = [
            {"src": "n0", "tgt": "n1", "rel": "adjacent_to"},
            {"src": "n1", "tgt": "n2", "rel": "interacting_with"},
        ]
        self_loops = [e for e in edges if e["src"] == e["tgt"]]
        self.assertEqual(len(self_loops), 0)

    def test_duplicate_edge_detection(self):
        edges = [
            {"src": "n0", "tgt": "n1", "rel": "adjacent_to"},
            {"src": "n0", "tgt": "n1", "rel": "adjacent_to"},
        ]
        seen = set(); dups = 0
        for e in edges:
            key = (e["src"], e["tgt"], e["rel"])
            if key in seen: dups += 1
            seen.add(key)
        self.assertEqual(dups, 1)


def run_unit_tests() -> dict:
    banner("STAGE G -- Regression / Unit Tests")
    loader = unittest.TestLoader()
    suite  = unittest.TestSuite()
    for cls in [TestIoU, TestLatencyTimer, TestGraphStructure]:
        suite.addTests(loader.loadTestsFromTestCase(cls))

    stream = io.StringIO()
    runner = unittest.TextTestRunner(stream=stream, verbosity=2)
    result = runner.run(suite)
    print(stream.getvalue())

    n_pass = result.testsRun - len(result.failures) - len(result.errors)
    n_fail = len(result.failures) + len(result.errors)

    if n_fail == 0: ok(f"All {result.testsRun} unit tests PASSED")
    else:           fail(f"{n_fail} unit test(s) FAILED")

    return {
        "tests_run": result.testsRun,
        "passed":    n_pass,
        "failed":    n_fail,
        "failures":  [str(f) for f in result.failures],
        "errors":    [str(e) for e in result.errors],
    }

print("Stage G function defined.")

In [ ]:
# ================================================================
#  CELL 13 - NEW: STAGE H -- MATPLOTLIB V&V DASHBOARD
#  6-panel figure: confusion matrix, P/R/F1, latency CDF,
#  detections/video, node-edge scatter, confidence histogram
# ================================================================

DASHBOARD_PATH = "validation_dashboard.png"

def generate_dashboard(full_report: dict, classes: list, out_path: str = DASHBOARD_PATH):
    banner("STAGE H -- V&V Dashboard")

    env_acc  = full_report.get("env_accuracy",    {})
    sgg_q    = full_report.get("sgg_quality",     {})
    sgg_sem  = full_report.get("sgg_semantics",   {})

    fig, axes = plt.subplots(2, 3, figsize=(18, 11))
    fig.suptitle("SGG Pipeline -- Validation & Verification Dashboard",
                 fontsize=15, fontweight="bold", y=1.01)
    sns.set_style("whitegrid")

    # H1 -- Confusion matrix
    ax = axes[0, 0]
    cm = env_acc.get("confusion_matrix")
    if cm:
        sns.heatmap(np.array(cm), annot=True, fmt="d", cmap="Blues",
                    xticklabels=classes, yticklabels=classes, ax=ax)
        ax.set_xlabel("Predicted"); ax.set_ylabel("True")
        ax.set_title("H1 -- Confusion Matrix")
    else:
        ax.text(0.5, 0.5, "No data", ha="center", va="center")
        ax.set_title("H1 -- Confusion Matrix (no data)")

    # H2 -- Per-class P/R/F1
    ax = axes[0, 1]
    pc = env_acc.get("per_class", {})
    if pc:
        cls_names = list(pc.keys())
        precs = [pc[c]["precision"] for c in cls_names]
        recs  = [pc[c]["recall"]    for c in cls_names]
        f1s   = [pc[c]["f1"]        for c in cls_names]
        x = np.arange(len(cls_names)); w = 0.25
        ax.bar(x - w, precs, w, label="Precision", color="#4C72B0")
        ax.bar(x,     recs,  w, label="Recall",    color="#DD8452")
        ax.bar(x + w, f1s,   w, label="F1",        color="#55A868")
        ax.set_xticks(x); ax.set_xticklabels(cls_names)
        ax.set_ylim(0, 1.1); ax.legend()
        ax.axhline(THRESH_CNN_ACCURACY, color="red", linestyle="--", linewidth=0.8)
        ax.set_title("H2 -- Per-class Precision / Recall / F1")
    else:
        ax.text(0.5, 0.5, "No data", ha="center", va="center")
        ax.set_title("H2 -- P/R/F1 (no data)")

    # H3 -- CNN latency CDF
    ax = axes[0, 2]
    lat = env_acc.get("latency_times", [])
    if lat:
        sorted_t = np.sort(lat)
        cdf      = np.arange(1, len(sorted_t)+1) / len(sorted_t)
        ax.plot(sorted_t, cdf, linewidth=2, color="#4C72B0")
        ax.axvline(np.percentile(sorted_t, 95), color="red", linestyle="--", label="p95")
        ax.set_xlabel("Latency (ms)"); ax.set_ylabel("CDF")
        ax.set_title("H3 -- CNN Inference Latency CDF")
        ax.legend()
    else:
        ax.text(0.5, 0.5, "No data", ha="center", va="center")
        ax.set_title("H3 -- CNN Latency CDF (no data)")

    # H4 -- Detections per video
    ax = axes[1, 0]
    vr = sgg_q.get("video_results", [])
    if vr:
        names_short = [Path(v["video"]).name[:15] for v in vr]
        det_counts  = [v["detections"] for v in vr]
        colors      = ["#55A868" if d > 0 else "#C44E52" for d in det_counts]
        ax.barh(range(len(names_short)), det_counts, color=colors)
        ax.set_yticks(range(len(names_short)))
        ax.set_yticklabels(names_short, fontsize=7)
        ax.set_xlabel("Total detections")
        ax.set_title("H4 -- Detections per Video")
    else:
        ax.text(0.5, 0.5, "No data", ha="center", va="center")
        ax.set_title("H4 -- Detections per Video (no data)")

    # H5 -- Node vs Edge scatter
    ax = axes[1, 1]
    nc = sgg_q.get("all_node_counts", [])
    ec = sgg_q.get("all_edge_counts", [])
    if nc and ec:
        ax.scatter(nc, ec, alpha=0.5, color="#4C72B0", edgecolors="white", s=40)
        if len(nc) > 2:
            z  = np.polyfit(nc, ec, 1)
            xs = np.linspace(min(nc), max(nc), 100)
            ax.plot(xs, np.poly1d(z)(xs), "r--", linewidth=1, label="trend")
        ax.set_xlabel("Nodes per frame"); ax.set_ylabel("Edges per frame")
        ax.set_title("H5 -- Nodes vs Edges per Frame")
        ax.legend()
    else:
        ax.text(0.5, 0.5, "No data", ha="center", va="center")
        ax.set_title("H5 -- Node/Edge Scatter (no data)")

    # H6 -- Confidence histogram
    ax = axes[1, 2]
    conf_scores = sgg_sem.get("all_conf_scores", [])
    th_data     = sgg_sem.get("threshold_sensitivity", {})
    if conf_scores:
        ax.hist(conf_scores, bins=30, color="#4C72B0", alpha=0.7, edgecolor="white")
        ax.axvline(YOLO_CONF_THRESHOLD, color="red", linestyle="--",
                   label=f"threshold={YOLO_CONF_THRESHOLD}")
        ax.set_xlabel("Confidence"); ax.set_ylabel("Count")
        ax.set_title("H6 -- YOLO Detection Confidence Histogram")
        ax.legend()
    elif th_data:
        ax.plot(th_data["thresholds"], th_data["detection_rates"], marker="o", linewidth=2)
        ax.set_xlabel("Confidence threshold"); ax.set_ylabel("Detection rate")
        ax.set_title("H6 -- Threshold Sensitivity")
    else:
        ax.text(0.5, 0.5, "No data", ha="center", va="center")
        ax.set_title("H6 -- Confidence (no data)")

    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    ok(f"Dashboard saved to {out_path}")

    # Separate threshold sensitivity plot
    if th_data:
        fig2, ax2 = plt.subplots(figsize=(7, 4))
        ax2.plot(th_data["thresholds"], th_data["detection_rates"],
                 marker="o", linewidth=2, color="#DD8452")
        ax2.axvline(YOLO_CONF_THRESHOLD, color="red",   linestyle="--", label=f"current={YOLO_CONF_THRESHOLD}")
        ax2.axhline(THRESH_DETECTION_RATE, color="green", linestyle=":",  label=f"required={THRESH_DETECTION_RATE}")
        ax2.set_xlabel("Confidence threshold")
        ax2.set_ylabel("Detection rate (frames with >=1 detection)")
        ax2.set_title("Threshold Sensitivity Analysis (Stage F/I)")
        ax2.legend(); ax2.grid(True)
        plt.savefig("threshold_sensitivity.png", dpi=150, bbox_inches="tight")
        plt.close(fig2)
        ok("Threshold sensitivity plot saved to threshold_sensitivity.png")

print("Stage H function defined.")

In [ ]:
# ================================================================
#  CELL 14 - DISCOVER VIDEOS & SAVE REPORT HELPERS
# ================================================================
def discover_test_videos(test_dir: str, classes: list) -> list:
    test_dir = Path(test_dir)
    results  = []
    for cls in classes:
        cls_dir = test_dir / cls
        if not cls_dir.exists():
            warn(f"No folder found for class '{cls}' at {cls_dir}")
            continue
        for ext in ("*.mp4", "*.avi", "*.mov", "*.mkv"):
            for vp in cls_dir.glob(ext):
                results.append((str(vp), cls))
    if results: ok(f"Discovered {len(results)} labelled test videos")
    else:       warn("No labelled test videos discovered")
    return results


def save_report(report: dict, output_path: str = REPORT_PATH):
    clean = json.loads(json.dumps(
        report,
        default=lambda o: float(o) if isinstance(o, (np.floating, np.integer))
                         else (o.tolist() if isinstance(o, np.ndarray) else str(o))
    ))
    with open(output_path, "w") as f:
        json.dump(clean, f, indent=2)
    ok(f"Full report saved to {output_path}")

print("Helper functions defined.")

In [ ]:
# ================================================================
#  CELL 15 - COLLECT TEST VIDEOS
# ================================================================
banner("Collecting Test Videos")
test_videos = []

if TEST_DIR:
    test_videos = discover_test_videos(TEST_DIR, CLASSES)

if SINGLE_VIDEO and SINGLE_LABEL:
    test_videos.append((SINGLE_VIDEO, SINGLE_LABEL))
    ok(f"Single video added: {SINGLE_VIDEO}  label={SINGLE_LABEL}")
elif SINGLE_VIDEO:
    warn("SINGLE_VIDEO provided without SINGLE_LABEL -- accuracy tests skipped for it.")
    test_videos.append((SINGLE_VIDEO, CLASSES[0]))

if not test_videos: warn("No test videos found -- smoke/unit tests will still run.")
else:               info(f"Total test videos: {len(test_videos)}")

In [ ]:
# ================================================================
#  CELL 16 - LOAD MODELS
# ================================================================
env_model, device = load_env_model(ENV_MODEL_PATH, num_classes=len(CLASSES))
sgg_model         = None
if not NO_SGG:
    sgg_model = load_sgg_model(YOLO_WEIGHTS, DEFAULT_SGG_VOCAB)

banner("SGG PIPELINE VALIDATION -- Starting")
print(f"  Model   : {ENV_MODEL_PATH}")
print(f"  Classes : {CLASSES}")
print(f"  Device  : {'GPU' if torch.cuda.is_available() else 'CPU'}")
print(f"  Time    : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

full_report = {
    "timestamp":  datetime.now().isoformat(),
    "model_path": ENV_MODEL_PATH,
    "classes":    CLASSES,
    "device":     "GPU" if torch.cuda.is_available() else "CPU",
}

In [ ]:
# ================================================================
#  CELL 17 - RUN ALL STAGES
# ================================================================

# Stage G - unit tests (no models or data required)
full_report["unit_tests"]     = run_unit_tests()

# Stage D - smoke tests
full_report["smoke_tests"]    = run_smoke_tests(env_model, device, sgg_model, CLASSES)

# Stage A - CNN accuracy
full_report["env_accuracy"]   = validate_env_model(env_model, device, test_videos, CLASSES)

# Stage B - SGG detection quality
full_report["sgg_quality"]    = validate_sgg(sgg_model, test_videos)

# Stage C - full pipeline latency
full_report["pipeline_latency"] = validate_pipeline_latency(
    env_model, device, sgg_model, test_videos, CLASSES)

# Stage E - SGG structural V&V  (NEW)
full_report["sgg_structure"]  = validate_sgg_structure(sgg_model, test_videos, DEFAULT_SGG_VOCAB)

# Stage F - SGG semantic V&V  (NEW)
full_report["sgg_semantics"]  = validate_sgg_semantics(sgg_model, test_videos, DEFAULT_SGG_VOCAB)

# Stage H - dashboard  (NEW)
generate_dashboard(full_report, CLASSES)

In [ ]:
# ================================================================
#  CELL 18 - NEW: STAGE J -- FINAL PASS/FAIL VERDICT
# ================================================================
banner("STAGE J -- Final Verification & Validation Verdict")

def verdict_row(criterion, value, threshold, fmt="{:.1%}", higher_is_better=True):
    """Print a coloured PASS/FAIL row and return status string."""
    if value is None:
        print(f"  {C.YELLOW}[SKIP]  {criterion:<45s}  N/A (no data){C.END}")
        return "SKIP"
    passes = (value >= threshold) if higher_is_better else (value <= threshold)
    col    = C.GREEN if passes else C.RED
    sym    = "PASS" if passes else "FAIL"
    print(f"  {col}[{sym}]  {criterion:<45s}  value={fmt.format(value)}  threshold={fmt.format(threshold)}{C.END}")
    return sym

statuses = []

ut = full_report.get("unit_tests", {})
statuses.append(verdict_row(
    "G  Unit tests (all pass)",
    ut.get("passed", 0) / ut["tests_run"] if ut.get("tests_run") else None,
    1.0))

sm = full_report.get("smoke_tests", {})
statuses.append(verdict_row(
    "D  Smoke test failures (must be 0)",
    float(sm.get("failed", 0)),
    0.0, fmt="{:.0f}", higher_is_better=False))

ea = full_report.get("env_accuracy", {})
statuses.append(verdict_row(
    "A  CNN top-1 accuracy",
    ea.get("accuracy"),
    THRESH_CNN_ACCURACY))

sq = full_report.get("sgg_quality", {})
statuses.append(verdict_row(
    "B  SGG detection rate",
    sq.get("detection_rate"),
    THRESH_DETECTION_RATE))

pl = full_report.get("pipeline_latency", {}).get("latency", {})
statuses.append(verdict_row(
    "C  Pipeline latency mean (ms/video)",
    pl.get("mean_ms"),
    THRESH_PIPELINE_MS, fmt="{:.0f}ms", higher_is_better=False))

es = full_report.get("sgg_structure", {})
statuses.append(verdict_row(
    "E  SGG triplet completeness",
    es.get("triplet_completeness"),
    THRESH_TRIPLET_COMPLETE))

statuses.append(verdict_row(
    "E  Self-loop edges (must be 0)",
    float(es["self_loop_count"]) if "self_loop_count" in es else None,
    0.0, fmt="{:.0f}", higher_is_better=False))

fs = full_report.get("sgg_semantics", {})
statuses.append(verdict_row(
    "F  SGG vocabulary compliance",
    fs.get("vocab_compliance"),
    0.95))

statuses.append(verdict_row(
    "F  Low-confidence FP rate",
    fs.get("low_conf_fp_rate"),
    0.20, higher_is_better=False))

n_pass = statuses.count("PASS")
n_fail = statuses.count("FAIL")
n_skip = statuses.count("SKIP")

banner(f"OVERALL: {n_pass} PASS  |  {n_fail} FAIL  |  {n_skip} SKIP")
if n_fail == 0:
    ok("Pipeline meets all V&V criteria -- READY FOR INTEGRATION")
else:
    fail(f"{n_fail} criterion(a) failed -- address before release")

full_report["verdict"] = {
    "pass": n_pass, "fail": n_fail, "skip": n_skip,
    "overall": "PASS" if n_fail == 0 else "FAIL",
    "criteria": statuses,
}

In [ ]:
# ================================================================
#  CELL 19 - SAVE REPORT
# ================================================================
save_report(full_report, REPORT_PATH)
banner("Validation Complete")
print(f"\n  Report saved to   : {REPORT_PATH}")
print(f"  Dashboard saved to: {DASHBOARD_PATH}")